# Phase 6 — Attention Head Analysis

Identify which of BERT's 144 attention heads (12 layers x 12 heads) matter for
each of the 23 active emotions.

**Methods.**
1. Head ablation: zero out every head (one at a time) and measure per-emotion F1 drop.
2. Importance map: heatmap of (head, emotion) importance scores.
3. Specialization: Gini-coefficient selectivity per head + quadrant categorisation.
4. Emotion-head affinity: top heads per emotion + hierarchical clustering.
5. Redundancy: correlation matrix between heads' emotion-importance profiles.
6. Attention pattern visualisation of the top specialists.
7. Layer-wise summary with compression recommendations.

**Pipeline.**
- **Setup** — paths, imports, helpers.
- **Part A** — load model + baseline F1.
- **Part B** — 144-head ablation + importance heatmaps.
- **Part C** — specialization (Gini + quadrants).
- **Part D** — emotion-head affinity + clustered heatmap.
- **Part E** — redundancy (head & layer correlations).
- **Part F** — attention pattern visualisation.
- **Part G** — layer-wise summary + export CSVs to `results/notebook6/`.

## Setup

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    os.chdir(PROJECT_ROOT)
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from src.data import load_goemotions
from src.models import load_bert_classifier

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
EXPORT_DIR = os.path.join(RESULTS_DIR, 'notebook6')
os.makedirs(EXPORT_DIR, exist_ok=True)

MODEL_SUBDIR = 'bert-goemotions-23emo-final'
MODEL_PATH = os.path.join(RESULTS_DIR, MODEL_SUBDIR)

EXCLUDED_EMOTIONS = ['neutral', 'grief', 'nervousness', 'pride', 'relief']

NUM_LAYERS = 12
NUM_HEADS = 12
HEAD_DIM = 64          # 768 / 12
NUM_HEADS_TOTAL = NUM_LAYERS * NUM_HEADS
N_EVAL = 2000          # subset size for ablation (full test set is too slow)
BATCH_SIZE = 64
TOP_HEADS_CLUSTER = 30

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'font.size': 11})

print(f'Project root: {PROJECT_ROOT}')
print(f'Device:       {DEVICE}')
print(f'Export dir:   {EXPORT_DIR}')

In [ ]:
@torch.no_grad()
def evaluate_per_emotion(model, loader, device):
    """Compute per-emotion F1 and macro F1."""
    all_preds, all_labels = [], []
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = (torch.sigmoid(outputs.logits.cpu()) > 0.5).float()
        all_preds.append(preds)
        all_labels.append(batch['labels'])
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    per_emotion = []
    for i in range(all_labels.shape[1]):
        if all_labels[:, i].sum() > 0:
            per_emotion.append(f1_score(all_labels[:, i], all_preds[:, i], zero_division=0))
        else:
            per_emotion.append(np.nan)
    return np.array(per_emotion), float(np.nanmean(per_emotion))


def ablate_head_via_attention_self(model, layer_idx, head_idx, head_dim=HEAD_DIM):
    """Register a forward hook that zeros out head_idx's slice in the attention output."""
    attn_self = model.bert.encoder.layer[layer_idx].attention.self

    def hook_fn(module, inputs, output):
        context = output[0]
        start = head_idx * head_dim
        end = (head_idx + 1) * head_dim
        modified = context.clone()
        modified[:, :, start:end] = 0.0
        return (modified,) + output[1:]

    return attn_self.register_forward_hook(hook_fn)


def gini_coefficient(values):
    """0 = uniform, 1 = fully concentrated."""
    values = np.abs(values)
    if values.sum() == 0:
        return 0.0
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * sorted_vals) / (n * sorted_vals.sum())) - (n + 1) / n

## Part A — Load Model and Baseline F1

In [ ]:
train_ds, val_ds, test_ds, emotion_names, data_collator = load_goemotions(
    exclude_emotions=EXCLUDED_EMOTIONS
)
NUM_EMOTIONS = len(emotion_names)

model = load_bert_classifier(model_name=MODEL_PATH, num_labels=NUM_EMOTIONS)
model.to(DEVICE).eval()

eval_subset = test_ds.select(range(min(N_EVAL, len(test_ds))))
eval_loader = DataLoader(eval_subset, batch_size=BATCH_SIZE, collate_fn=data_collator, shuffle=False)

print(f'Model loaded  |  {NUM_LAYERS}L x {NUM_HEADS}H = {NUM_HEADS_TOTAL} heads')
print(f'Emotions: {NUM_EMOTIONS}')
print(f'Ablation subset: {len(eval_subset)} samples')

In [ ]:
baseline_per_emotion, baseline_macro = evaluate_per_emotion(model, eval_loader, DEVICE)
print(f'Baseline macro F1: {baseline_macro:.4f}')
print('\nPer-emotion F1:')
for i, name in enumerate(emotion_names):
    print(f'  {name:20s}: {baseline_per_emotion[i]:.4f}')

## Part B — Head Ablation Study

In [ ]:
importance_matrix = np.zeros((NUM_HEADS_TOTAL, NUM_EMOTIONS))
macro_drops = np.zeros(NUM_HEADS_TOTAL)

pbar = tqdm(total=NUM_HEADS_TOTAL, desc='Ablating heads')
for layer_idx in range(NUM_LAYERS):
    for head_idx in range(NUM_HEADS):
        flat_idx = layer_idx * NUM_HEADS + head_idx
        handle = ablate_head_via_attention_self(model, layer_idx, head_idx)
        ablated_per_emo, ablated_macro = evaluate_per_emotion(model, eval_loader, DEVICE)
        handle.remove()

        importance_matrix[flat_idx] = baseline_per_emotion - ablated_per_emo
        macro_drops[flat_idx] = baseline_macro - ablated_macro
        pbar.set_postfix({'L': layer_idx, 'H': head_idx,
                          'macro_drop': f'{macro_drops[flat_idx]:.4f}'})
        pbar.update(1)
pbar.close()

records = []
for layer_idx in range(NUM_LAYERS):
    for head_idx in range(NUM_HEADS):
        flat_idx = layer_idx * NUM_HEADS + head_idx
        rec = {
            'layer': layer_idx, 'head': head_idx, 'flat_idx': flat_idx,
            'macro_f1_drop': macro_drops[flat_idx],
        }
        for e_idx, e_name in enumerate(emotion_names):
            rec[f'f1_drop_{e_name}'] = importance_matrix[flat_idx, e_idx]
        records.append(rec)
ablation_df = pd.DataFrame(records)

print(f'Importance matrix shape: {importance_matrix.shape}')
print('\nTop 10 heads by macro F1 drop:')
for _, row in ablation_df.nlargest(10, 'macro_f1_drop').iterrows():
    print(f'  L{int(row["layer"])}H{int(row["head"])}: macro drop = {row["macro_f1_drop"]:+.4f}')

### B.1 — Mean Head Importance Heatmap (Layer x Head)

In [ ]:
importance_3d = importance_matrix.reshape(NUM_LAYERS, NUM_HEADS, NUM_EMOTIONS)
mean_importance = importance_3d.mean(axis=2)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    mean_importance, annot=True, fmt='.3f',
    xticklabels=[f'H{i}' for i in range(NUM_HEADS)],
    yticklabels=[f'L{i}' for i in range(NUM_LAYERS)],
    cmap='RdYlBu_r', center=0, ax=ax,
)
ax.set_xlabel('Head')
ax.set_ylabel('Layer')
ax.set_title('Mean Head Importance (F1 Drop When Ablated)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'head_importance_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

### B.2 — Per-Emotion Importance Grid

In [ ]:
n_cols = 7
n_rows = int(np.ceil(NUM_EMOTIONS / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()
for e_idx in range(NUM_EMOTIONS):
    sns.heatmap(
        importance_3d[:, :, e_idx],
        cmap='RdYlBu_r', center=0,
        xticklabels=False, yticklabels=False, cbar=False, ax=axes[e_idx],
    )
    axes[e_idx].set_title(emotion_names[e_idx], fontsize=9)
for j in range(NUM_EMOTIONS, len(axes)):
    axes[j].axis('off')

plt.suptitle('Head Importance per Emotion (F1 Drop)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'head_importance_per_emotion.png'), dpi=150, bbox_inches='tight')
plt.show()

## Part C — Head Specialization

**Selectivity** = Gini coefficient of each head's |F1 drop| vector across emotions.
- High selectivity + high importance = **critical specialist**.
- Low selectivity + high importance = **critical generalist**.
- High selectivity + low importance = **minor specialist**.
- Low selectivity + low importance = **dispensable**.

In [ ]:
head_selectivity = np.zeros(NUM_HEADS_TOTAL)
head_total_importance = np.zeros(NUM_HEADS_TOTAL)
head_top_emotions = [None] * NUM_HEADS_TOTAL

for flat_idx in range(NUM_HEADS_TOTAL):
    imp = importance_matrix[flat_idx]
    head_selectivity[flat_idx] = gini_coefficient(imp)
    head_total_importance[flat_idx] = np.abs(imp).sum()
    top_3 = np.argsort(imp)[::-1][:3]
    head_top_emotions[flat_idx] = [(emotion_names[i], imp[i]) for i in top_3]

print(f'Selectivity range:      [{head_selectivity.min():.3f}, {head_selectivity.max():.3f}]')
print(f'Total importance range: [{head_total_importance.min():.3f}, {head_total_importance.max():.3f}]')

### C.1 — Selectivity vs Importance Scatter

In [ ]:
layers_flat = np.array([flat_idx // NUM_HEADS for flat_idx in range(NUM_HEADS_TOTAL)])

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    head_total_importance, head_selectivity,
    c=layers_flat, cmap='viridis', alpha=0.7, s=60,
    edgecolors='white', linewidth=0.5,
)

imp_q95 = np.percentile(head_total_importance, 95)
sel_q90 = np.percentile(head_selectivity, 90)
imp_q50 = np.percentile(head_total_importance, 50)
for flat_idx in range(NUM_HEADS_TOTAL):
    layer = flat_idx // NUM_HEADS
    head = flat_idx % NUM_HEADS
    imp = head_total_importance[flat_idx]
    sel = head_selectivity[flat_idx]
    if imp > imp_q95 or (sel > sel_q90 and imp > imp_q50):
        ax.annotate(f'L{layer}H{head}', (imp, sel), fontsize=7, ha='center', va='bottom')

plt.colorbar(scatter, ax=ax, label='Layer')
ax.set_xlabel('Total Importance (sum of |F1 drops|)')
ax.set_ylabel('Selectivity (Gini coefficient)')
ax.set_title('Head Specialization: Importance vs Selectivity')
ax.axhline(y=np.median(head_selectivity), color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=np.median(head_total_importance), color='gray', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'head_specialization_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

### C.2 — Quadrant Categorisation

In [ ]:
med_imp = np.median(head_total_importance)
med_sel = np.median(head_selectivity)

CATEGORIES = ['Critical Specialists', 'Critical Generalists', 'Minor Specialists', 'Dispensable']
categories = {k: [] for k in CATEGORIES}

for flat_idx in range(NUM_HEADS_TOTAL):
    layer = flat_idx // NUM_HEADS
    head = flat_idx % NUM_HEADS
    imp = head_total_importance[flat_idx]
    sel = head_selectivity[flat_idx]
    hi_imp = imp > med_imp
    hi_sel = sel > med_sel
    if hi_imp and hi_sel:
        cat = 'Critical Specialists'
    elif hi_imp and not hi_sel:
        cat = 'Critical Generalists'
    elif not hi_imp and hi_sel:
        cat = 'Minor Specialists'
    else:
        cat = 'Dispensable'
    categories[cat].append((layer, head, imp, sel))

print('Head Categories:')
print('=' * 60)
for cat_name, heads in categories.items():
    print(f'\n{cat_name} ({len(heads)} heads):')
    for layer, head, imp, sel in sorted(heads, key=lambda x: x[2], reverse=True)[:5]:
        top_em = head_top_emotions[layer * NUM_HEADS + head]
        top_str = ', '.join([f'{e}({v:+.3f})' for e, v in top_em])
        print(f'  L{layer}H{head}: imp={imp:.3f}, sel={sel:.3f} | top: {top_str}')

## Part D — Emotion-Head Affinity

In [ ]:
emotion_head_mapping = {}
print('Critical Heads per Emotion (top 5 by F1 drop):')
print('=' * 70)
for e_idx, e_name in enumerate(emotion_names):
    imp = importance_matrix[:, e_idx]
    top_5 = np.argsort(imp)[::-1][:5]
    critical = []
    print(f'\n{e_name} (baseline F1={baseline_per_emotion[e_idx]:.3f}):')
    for idx in top_5:
        layer = idx // NUM_HEADS
        head = idx % NUM_HEADS
        drop = imp[idx]
        print(f'  L{layer}H{head}: F1 drop = {drop:+.4f}')
        critical.append((int(layer), int(head), float(drop)))
    emotion_head_mapping[e_name] = critical

### D.1 — Emotion-Head Clustermap (Top 30 Heads)

In [ ]:
overall_importance = np.abs(importance_matrix).sum(axis=1)
top_head_indices = np.argsort(overall_importance)[::-1][:TOP_HEADS_CLUSTER]

sub_matrix = importance_matrix[top_head_indices].T
head_labels = [f'L{i // NUM_HEADS}H{i % NUM_HEADS}' for i in top_head_indices]

g = sns.clustermap(
    pd.DataFrame(sub_matrix, index=emotion_names, columns=head_labels),
    cmap='RdYlBu_r', center=0, figsize=(16, 10),
    row_cluster=True, col_cluster=True,
    dendrogram_ratio=(0.1, 0.1),
    cbar_kws={'label': 'F1 Drop (positive = head important)'},
)
g.fig.suptitle(f'Emotion-Head Affinity (Top {TOP_HEADS_CLUSTER} Heads)',
               y=1.02, fontsize=14)
plt.savefig(os.path.join(RESULTS_DIR, 'emotion_head_clustermap.png'), dpi=150, bbox_inches='tight')
plt.show()

## Part E — Redundancy

Two heads are redundant if their emotion-importance profiles are strongly
correlated; ablating either alone barely hurts, but ablating both should.
Pearson correlation acts as a first-order approximation.

In [ ]:
head_correlation = np.corrcoef(importance_matrix)

redundant_pairs = []
for i in range(NUM_HEADS_TOTAL):
    for j in range(i + 1, NUM_HEADS_TOTAL):
        corr = head_correlation[i, j]
        if not np.isnan(corr):
            redundant_pairs.append((i, j, corr))
redundant_pairs.sort(key=lambda x: x[2], reverse=True)

print('Top 15 most redundant pairs (highest correlation):')
print('=' * 60)
for i, j, corr in redundant_pairs[:15]:
    l1, h1 = i // NUM_HEADS, i % NUM_HEADS
    l2, h2 = j // NUM_HEADS, j % NUM_HEADS
    print(f'  L{l1}H{h1} <-> L{l2}H{h2}: r = {corr:.4f}')

### E.1 — Inter-layer Correlation Heatmap

In [ ]:
layer_corr = np.zeros((NUM_LAYERS, NUM_LAYERS))
for l1 in range(NUM_LAYERS):
    for l2 in range(NUM_LAYERS):
        block = head_correlation[l1*NUM_HEADS:(l1+1)*NUM_HEADS,
                                 l2*NUM_HEADS:(l2+1)*NUM_HEADS]
        if l1 == l2:
            mask = ~np.eye(NUM_HEADS, dtype=bool)
            layer_corr[l1, l2] = np.nanmean(block[mask])
        else:
            layer_corr[l1, l2] = np.nanmean(block)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    layer_corr, annot=True, fmt='.2f',
    xticklabels=[f'L{i}' for i in range(NUM_LAYERS)],
    yticklabels=[f'L{i}' for i in range(NUM_LAYERS)],
    cmap='RdYlBu_r', center=0, vmin=-1, vmax=1, ax=ax,
)
ax.set_title('Mean Head Importance Correlation Between Layers')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'head_redundancy_layers.png'), dpi=150, bbox_inches='tight')
plt.show()

## Part F — Attention Patterns of Top Specialists

Load an eager-attention copy of the model to extract attention weights
(`sdpa` does not support `output_attentions=True`) and visualise the top
specialist heads on example sentences.

In [ ]:
specialist_score = head_selectivity * head_total_importance
top_specialist_indices = np.argsort(specialist_score)[::-1][:6]

print('Top 6 specialist heads:')
for idx in top_specialist_indices:
    layer = idx // NUM_HEADS
    head = idx % NUM_HEADS
    top_em = head_top_emotions[idx]
    print(f'  L{layer}H{head}: selectivity={head_selectivity[idx]:.3f}, '
          f'importance={head_total_importance[idx]:.3f}')
    print(f'    Top emotions: {", ".join([f"{e}({v:+.3f})" for e, v in top_em])}')

In [ ]:
EXAMPLE_SENTENCES = [
    'I am so happy and grateful for everything you\'ve done!',
    'This is absolutely disgusting and makes me angry.',
    "I'm really scared about what might happen next.",
    "That's hilarious, I can't stop laughing!",
    'I feel so sad and lonely right now.',
    "Wow, I'm genuinely surprised by this news.",
]

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
eager_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=NUM_EMOTIONS,
    problem_type='multi_label_classification',
    attn_implementation='eager',
)
eager_model.to(DEVICE).eval()


@torch.no_grad()
def get_attention_weights(text, tokenizer, device):
    inputs = tokenizer(text, return_tensors='pt', padding=True,
                       truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = eager_model.bert(**inputs, output_attentions=True)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].cpu())
    return outputs.attentions, tokens


n_specialist = min(4, len(top_specialist_indices))
n_sentences = min(3, len(EXAMPLE_SENTENCES))

fig, axes = plt.subplots(n_specialist, n_sentences,
                         figsize=(6 * n_sentences, 5 * n_specialist))
if n_specialist == 1:
    axes = axes.reshape(1, -1)
if n_sentences == 1:
    axes = axes.reshape(-1, 1)

for s_idx, head_flat_idx in enumerate(top_specialist_indices[:n_specialist]):
    layer = head_flat_idx // NUM_HEADS
    head = head_flat_idx % NUM_HEADS
    top_em = head_top_emotions[head_flat_idx]
    for sent_idx in range(n_sentences):
        attentions, tokens = get_attention_weights(
            EXAMPLE_SENTENCES[sent_idx], tokenizer, DEVICE
        )
        attn = attentions[layer][0, head].cpu().numpy()
        n_tokens = len(tokens)
        attn = attn[:n_tokens, :n_tokens]
        ax = axes[s_idx, sent_idx]
        sns.heatmap(attn, xticklabels=tokens, yticklabels=tokens,
                    cmap='Blues', ax=ax, cbar=False, square=True)
        ax.set_title(f'L{layer}H{head} ({top_em[0][0]})', fontsize=9)
        ax.tick_params(axis='both', labelsize=6)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        plt.setp(ax.get_yticklabels(), rotation=0)

plt.suptitle('Attention Patterns of Top Specialist Heads', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'specialist_head_attention.png'), dpi=150, bbox_inches='tight')
plt.show()

## Part G — Layer-wise Summary + Compression Recommendations

In [ ]:
layer_records = []
for l in range(NUM_LAYERS):
    block = importance_matrix[l*NUM_HEADS:(l+1)*NUM_HEADS]
    layer_records.append({
        'layer': l,
        'mean_abs_importance': float(np.abs(block).mean()),
        'max_abs_importance': float(np.abs(block).max()),
        'mean_head_importance': float(head_total_importance[l*NUM_HEADS:(l+1)*NUM_HEADS].mean()),
        'max_head_importance': float(head_total_importance[l*NUM_HEADS:(l+1)*NUM_HEADS].max()),
        'mean_selectivity':    float(head_selectivity[l*NUM_HEADS:(l+1)*NUM_HEADS].mean()),
        'n_positive_heads': int((block.mean(axis=1) > 0).sum()),
        'n_negative_heads': int((block.mean(axis=1) < 0).sum()),
    })
layer_df = pd.DataFrame(layer_records)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(layer_df['layer'], layer_df['mean_abs_importance'], color='steelblue')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Mean |F1 Drop|')
axes[0].set_title('Layer-wise Head Importance')
axes[0].set_xticks(range(NUM_LAYERS))

box_rows = [{'Layer': f'L{l}',
             'Mean F1 Drop': importance_matrix[l*NUM_HEADS + h].mean()}
            for l in range(NUM_LAYERS) for h in range(NUM_HEADS)]
sns.boxplot(data=pd.DataFrame(box_rows), x='Layer', y='Mean F1 Drop',
            ax=axes[1], color='steelblue')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Head Importance Distribution by Layer')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'layer_head_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

imp_q75 = np.percentile(head_total_importance, 75)
imp_q25 = np.percentile(head_total_importance, 25)
print('\nPer-layer attention compression recommendation:')
for rec in layer_records:
    m = rec['mean_head_importance']
    if m > imp_q75:
        label = 'PRESERVE (high rank)'
    elif m > imp_q25:
        label = 'MODERATE compression'
    else:
        label = 'AGGRESSIVE compression (low rank)'
    print(f'  Layer {rec["layer"]:2d}: mean_imp={m:.4f}, max_imp={rec["max_head_importance"]:.4f} -> {label}')

disp_layers = [h[0] for h in categories['Dispensable']]
print(f'\nDispensable heads per layer: {dict(sorted(Counter(disp_layers).items()))}')

## Part H — Export

All results as CSV under `results/notebook6/`:

- `head_ablation_results.csv` — wide (144 x (24+)): per-head macro drop + per-emotion columns.
- `head_ablation_long.csv` — long (144 x 23 rows): (layer, head, emotion, f1_drop).
- `head_importance_matrix.csv` — wide matrix (144 x 23) flat indexed.
- `head_categories.csv` — quadrant membership per head with importance & selectivity.
- `head_top_emotions.csv` — top 3 emotions per head (long form).
- `top_heads_per_emotion.csv` — critical 5 heads per emotion.
- `head_correlation_matrix.csv` — 144 x 144 Pearson correlation of importance profiles.
- `head_redundancy_pairs.csv` — all unique head pairs sorted by correlation.
- `layer_redundancy_matrix.csv` — 12 x 12 mean inter-layer correlation.
- `layer_attention_importance.csv` — per-layer summary.
- `head_analysis_summary.csv` — one-row master summary.

In [ ]:
exports = {}

# 1. Ablation wide
exports['head_ablation_results.csv'] = ablation_df

# 2. Ablation long
rows_long = []
for flat_idx in range(NUM_HEADS_TOTAL):
    layer = flat_idx // NUM_HEADS
    head = flat_idx % NUM_HEADS
    for e_idx, e_name in enumerate(emotion_names):
        rows_long.append({
            'layer': layer, 'head': head, 'flat_idx': flat_idx,
            'emotion': e_name, 'f1_drop': float(importance_matrix[flat_idx, e_idx]),
            'baseline_f1': float(baseline_per_emotion[e_idx]),
        })
exports['head_ablation_long.csv'] = pd.DataFrame(rows_long)

# 3. Importance matrix wide (144 rows x 23 emotions)
imp_wide = pd.DataFrame(importance_matrix, columns=emotion_names)
imp_wide.insert(0, 'head', [i % NUM_HEADS for i in range(NUM_HEADS_TOTAL)])
imp_wide.insert(0, 'layer', [i // NUM_HEADS for i in range(NUM_HEADS_TOTAL)])
exports['head_importance_matrix.csv'] = imp_wide

# 4. Categories
cat_records = []
for cat_name, heads in categories.items():
    for layer, head, imp, sel in heads:
        cat_records.append({
            'layer': layer, 'head': head, 'category': cat_name,
            'total_importance': imp, 'selectivity': sel,
            'macro_f1_drop': macro_drops[layer * NUM_HEADS + head],
        })
exports['head_categories.csv'] = pd.DataFrame(cat_records)

# 5. Top emotions per head
rows_tophead = []
for flat_idx in range(NUM_HEADS_TOTAL):
    layer = flat_idx // NUM_HEADS
    head = flat_idx % NUM_HEADS
    for rank, (emo, drop) in enumerate(head_top_emotions[flat_idx], start=1):
        rows_tophead.append({
            'layer': layer, 'head': head, 'rank': rank,
            'emotion': emo, 'f1_drop': float(drop),
        })
exports['head_top_emotions.csv'] = pd.DataFrame(rows_tophead)

# 6. Top heads per emotion
rows_topemo = []
for e_name, critical in emotion_head_mapping.items():
    for rank, (layer, head, drop) in enumerate(critical, start=1):
        rows_topemo.append({
            'emotion': e_name, 'rank': rank,
            'layer': layer, 'head': head, 'f1_drop': drop,
        })
exports['top_heads_per_emotion.csv'] = pd.DataFrame(rows_topemo)

# 7. Head correlation matrix (144 x 144)
head_id = [f'L{i // NUM_HEADS}H{i % NUM_HEADS}' for i in range(NUM_HEADS_TOTAL)]
corr_df = pd.DataFrame(head_correlation, index=head_id, columns=head_id)
corr_df.index.name = 'head'
exports['head_correlation_matrix.csv'] = corr_df.reset_index()

# 8. Redundancy pairs (all pairs)
rows_red = []
for i, j, corr in redundant_pairs:
    rows_red.append({
        'head_a_layer': i // NUM_HEADS, 'head_a_head': i % NUM_HEADS,
        'head_b_layer': j // NUM_HEADS, 'head_b_head': j % NUM_HEADS,
        'correlation': float(corr),
    })
exports['head_redundancy_pairs.csv'] = pd.DataFrame(rows_red)

# 9. Layer redundancy
lr_df = pd.DataFrame(layer_corr,
                     index=[f'L{i}' for i in range(NUM_LAYERS)],
                     columns=[f'L{i}' for i in range(NUM_LAYERS)])
lr_df.index.name = 'layer'
exports['layer_redundancy_matrix.csv'] = lr_df.reset_index()

# 10. Layer attention importance
exports['layer_attention_importance.csv'] = layer_df

# 11. Master summary
top_macro_idx = int(np.argmax(macro_drops))
exports['head_analysis_summary.csv'] = pd.DataFrame([{
    'n_heads_total': NUM_HEADS_TOTAL,
    'n_emotions': NUM_EMOTIONS,
    'n_eval_samples': len(eval_subset),
    'baseline_macro_f1': baseline_macro,
    'most_critical_head': f'L{top_macro_idx // NUM_HEADS}H{top_macro_idx % NUM_HEADS}',
    'most_critical_head_macro_drop': float(macro_drops[top_macro_idx]),
    'n_critical_specialists': len(categories['Critical Specialists']),
    'n_critical_generalists': len(categories['Critical Generalists']),
    'n_minor_specialists':    len(categories['Minor Specialists']),
    'n_dispensable':          len(categories['Dispensable']),
    'top_redundant_pair':     f'L{redundant_pairs[0][0] // NUM_HEADS}H{redundant_pairs[0][0] % NUM_HEADS}-'
                              f'L{redundant_pairs[0][1] // NUM_HEADS}H{redundant_pairs[0][1] % NUM_HEADS}',
    'top_redundant_correlation': float(redundant_pairs[0][2]),
}])

for fname, df in exports.items():
    out_path = os.path.join(EXPORT_DIR, fname)
    df.to_csv(out_path, index=False)
    print(f'  [{len(df):>5d} rows] {fname}')

# Also keep the numpy matrix (legacy consumers)
np.save(os.path.join(EXPORT_DIR, 'head_importance_matrix.npy'), importance_matrix)

print('\n' + '=' * 70)
print('HEAD ANALYSIS SUMMARY - Implications for Informed Compression')
print('=' * 70)
print(f'Baseline macro F1: {baseline_macro:.4f}')
print(f'\nHead category counts:')
for cat_name in CATEGORIES:
    print(f'  {cat_name:25s}: {len(categories[cat_name]):3d} heads')
print(f'\nExported {len(exports)} CSVs (+1 .npy) to {EXPORT_DIR}')